In [1]:
import os
import pickle
import plotly.io as pio
pio.renderers.default = "notebook_connected"
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

In [3]:
# Charger les données

data_path = "../data/processed"
file_name = 'dataset_final_phrases.pkl'
with open(os.path.join(data_path, file_name), 'rb') as f:
    df = pickle.load(f)

In [4]:
# Charger un seul embedding

embeddings_path = "../data/embeddings"
# file_name = 'emb_sbert_multi.pkl'

# with open(os.path.join(embeddings_path, file_name), 'rb') as f:
#     embedding = pickle.load(f)

In [5]:
# Charger tous les embeddings

# embeddings_path = "../data/embeddings"
# embeddings = {}

# for file_name in os.listdir(embeddings_path):
#     with open(os.path.join(embeddings_path, file_name), 'rb') as f:
#         emb_name = file_name.split('.')[0]
#         embeddings[emb_name] = pickle.load(f)

### BERTopic

In [6]:
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
import nltk
from nltk.corpus import stopwords
from hdbscan import HDBSCAN
import umap
import numpy as np
from sklearn.cluster import KMeans
import pandas as pd
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import normalize
import warnings

/mnt/c/Users/charb/Documents/ALTERNANCE/Datascientest/Projet/TrustPilot/DS/trustpilot/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning:

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html



In [7]:
sentences = df["sentence"].tolist()

file_name_mutli = 'emb_sbert_multi_phrases.pkl'
with open(os.path.join(embeddings_path, file_name_mutli), 'rb') as f:
    embedding_multi = pickle.load(f)

file_name_fr = 'emb_sbert_fr_phrases.pkl'
with open(os.path.join(embeddings_path, file_name_fr), 'rb') as f:
    embedding_fr = pickle.load(f)

file_name_perf = 'emb_sbert_multi2_phrases.pkl'
with open(os.path.join(embeddings_path, file_name_fr), 'rb') as f:
    embedding_multi2 = pickle.load(f)

In [8]:
print(len(sentences))
print(embedding_multi.shape)
print(embedding_fr.shape)
print(embedding_multi2.shape)

37628
(37628, 384)
(37628, 768)
(37628, 768)


In [16]:
warnings.filterwarnings("ignore", category=UserWarning)

product_words = ['montre', 'montres', 'boucle', 'boucles', 'oreilles', 'oreille', 'paire', 'paires', 'lunettes', 'bracelet', 'bracelets', 'collier', 'iphone', 'téléphone', 'pendentif', 'robot', 'robots', 'aspirateur', 'baskets', 'basket', 'chaussure', 'chaussures', 'sandales', 'plantes', 'plante', 'arbres', 'arbre', 'bulbes', 'willemse', 'sommiers', 'jardin', 'bague', 'lampe', 'lampes', 'abat-jour', 'abat jour', 'lampadaire', 'parfum', 'shampoing', 'shampooing', 'shampooings', 'cheveux', 'masque', 'masques', 'crème', 'élastiques', 'manteau', 'bougie', 'cadre', 'écouteurs', 'vélo', 'robe', 'vêtements', 'bijoux', 'sac', 'portable', 'clio', 'luminaire', 'oreillette', 'induction', 'écouteur', 'couette', 'samsung', 'téléphones', 'smartcase', 'abat', 'apple', 'watch', 'shirt', 'tee', 'chemise', 'shirts', 'hortensias', 'orchidée', 'sacs', 'plant', 'reconditionné']

models = {}

for embedding, name in zip([embedding_fr, embedding_multi, embedding_multi2], ["français", "multilingue", "multilingue_2"]):
    print(name)

    vectorizer_model = CountVectorizer(
        stop_words=stopwords.words("french") + product_words,
        ngram_range=(1, 3),
        #min_df=2,
        max_df=0.95
    )
    # ctfidf_model = ClassTfidfTransformer()

    hdbscan_model = HDBSCAN(
    min_cluster_size=10,
    min_samples=3
    )
    
    topic_model = BERTopic(
        language="french",
        vectorizer_model=vectorizer_model,
        hdbscan_model=hdbscan_model,
        verbose=False,
        nr_topics="auto"
        # ctfidf_model=ctfidf_model
    )
    
    topics, probs = topic_model.fit_transform(sentences, embedding)

    # topic_model.reduce_topics(sentences, nr_topics=30)
    
    topics = np.array(topics)
    
    # Identifier les gros clusters
    topic_sizes = pd.Series(topics).value_counts()
    large_topics = topic_sizes[topic_sizes > 2000].index  # seuil à ajuster selon dataset
    
    for t in large_topics:
        print("Topic :", t)
        # Récupérer les indices des documents dans le gros cluster
        idx = np.where(topics == t)[0]
        embeddings_subset = embedding[idx]
        
        # Diviser le gros cluster avec KMeans
        n_subclusters = int(len(idx) / 200)  # ~200 docs par sous-cluster
        print("Nombre de clusters créés par k-means :", n_subclusters)
        
        emb_norm = normalize(embeddings_subset)
        kmeans = MiniBatchKMeans(
            n_clusters=n_subclusters,
            batch_size=512,
            max_iter=200,
            n_init="auto"
        )
        sub_labels = kmeans.fit_predict(emb_norm)
        
        # Réassigner les labels dans `topics`
        max_topic_id = topics.max() + 1
        for i, doc_idx in enumerate(idx):
            topics[doc_idx] = max_topic_id + sub_labels[i]

        topic_model.update_topics(
            docs=sentences,
            topics=topics,
            vectorizer_model=topic_model.vectorizer_model,
            top_n_words=15
        )

    models[name] = topic_model

français
Topic : -1
Nombre de clusters créés par k-means : 111


2025-11-21 17:53:50,861 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Topic : 0
Nombre de clusters créés par k-means : 68


2025-11-21 17:53:54,204 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


multilingue
Topic : 0
Nombre de clusters créés par k-means : 100


2025-11-21 17:54:07,677 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.
2025-11-21 17:54:10,929 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Topic : -1
Nombre de clusters créés par k-means : 80
multilingue_2
Topic : -1
Nombre de clusters créés par k-means : 113


2025-11-21 17:54:30,992 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.
2025-11-21 17:54:34,349 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Topic : 0
Nombre de clusters créés par k-means : 44


### Evaluation

#### Diversité

In [48]:
from itertools import chain

def topic_diversity(topic_model, top_n=10):
    """
    Calcule la diversité des topics d'un modèle BERTopic.
    """
    topics = topic_model.get_topics()

    # On extrait les mots uniquement (sans les scores)
    topic_words = []
    for topic_id, word_scores in topics.items():
        # BERTopic place les topics -1 et autres meta-topics, donc on ignore topic -1
        if topic_id == -1:
            continue
        top_words = [w for (w, score) in word_scores[:top_n]]
        topic_words.append(top_words)

    # Liste aplatie
    all_words = list(chain.from_iterable(topic_words))
    unique_words = set(all_words)

    return len(unique_words) / len(all_words)


# Calcul du score pour chaque modèle
diversity_scores = {}

for name, model in models.items():
    score = topic_diversity(model, top_n=10)
    diversity_scores[name] = score

diversity_scores

{'français': 0.6946428571428571,
 'multilingue': 0.7632692307692308,
 'multilingue_2': 0.5084577114427861}

#### Score de cohérence

In [49]:
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary
from collections import defaultdict
import numpy as np
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

for name, model in models.items():
    topics = model.topics_
    docs_by_topic = defaultdict(list)
    for doc, t in zip(sentences, topics):
        if t != -1:
            docs_by_topic[t].append(doc)
    
    # Générer top words manuellement
    topic_words = []
    for t, docs in docs_by_topic.items():
        vec = TfidfVectorizer(stop_words=stopwords.words("french") + product_words, ngram_range=(1,3))
        X = vec.fit_transform(docs)
        feature_names = np.array(vec.get_feature_names_out())
        # tfidf_sum = X.toarray().sum(axis=0)
        tfidf_sum = np.asarray(X.sum(axis=0)).ravel()
        top_words = feature_names[np.argsort(tfidf_sum)[::-1]][:10].tolist()
        topic_words.append(top_words)
    
    # Tokenisation des documents
    tokenized_docs = [doc.lower().split() for doc in sentences]
    dictionary = Dictionary(tokenized_docs)
    
    # Calcul du score de cohérence
    cm = CoherenceModel(
        topics=topic_words,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_v'
    )
    score = cm.get_coherence()
    print(name, score)

français 0.5219372121080242
multilingue 0.5077983464117374
multilingue_2 0.5430946881683159


#### Embedding-based coherence score

In [50]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def embedding_coherence(topic_model, embedder, top_n=10):
    """
    Calcule la cohérence basée sur les embeddings pour un modèle BERTopic.
    embedder : modèle sentence-transformers pour transformer les mots en vecteurs.
    """
    topics = topic_model.get_topics()
    scores = []

    for topic_id, word_scores in topics.items():
        if topic_id == -1:
            continue
        top_words = [w for w, _ in word_scores[:top_n]]
        word_embeddings = embedder.encode(top_words)
        sim_matrix = cosine_similarity(word_embeddings)
        
        # On enlève la diagonale (sim = 1)
        n = len(top_words)
        if n > 1:
            sims = (sim_matrix.sum() - n) / (n*(n-1))  # moyenne des cosinus
            scores.append(sims)

    return np.mean(scores)

# Exemple avec un modèle SentenceTransformer
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer('all-MiniLM-L6-v2')

for name, model in models.items():
    score = embedding_coherence(model, embedder, top_n=10)
    print(f"{name} - Embedding-based coherence: {score:.4f}")

français - Embedding-based coherence: 0.4020
multilingue - Embedding-based coherence: 0.4044
multilingue_2 - Embedding-based coherence: 0.3553


### Stockage

In [60]:
# for name, model in models.items():
#     with open(f"../models/topic_modeling/bertopic_{name}_phrases.pkl", "wb") as f:
#         pickle.dump(model, f)

In [61]:
# df["topics"] = models["multilingue_2"].topics_
# df.to_csv("./artifacts/bertopic/reviews_phrases_with_topics.csv", index=False, encoding="utf8")

In [7]:
import pandas as pd

# Charger le fichier CSV
df_avis = pd.read_csv("./artifacts/bertopic/reviews_phrases_with_topics.csv")

# Charger le fichier excel avec les nouveaux groupes
df_clusters = pd.read_excel("./artifacts/bertopic/topics_top_words_phrases_annoté.xlsx")

# Fusionner les fichiers sur la colonne du cluster
df_merged = df_avis.merge(
    df_clusters[["Topic", "Category"]],
    left_on='topics', # nom dans le CSV
    right_on='Topic', # nom dans l'Excel
    how='left'
)
df_merged = df_merged.drop(columns=['Topic'])

df_merged = df_merged[df_merged['Category'] != 'neutre']

map_clusters = {
    "qualité produit": 0,
    "livraison": 1,
    "service client": 2
}

df_merged['Label'] = df_merged['Category'].map(map_clusters)

# Exporter la nouvelle version
# df_merged.to_csv("../resultats/bertopic/data/dataset_labellise_phrases.csv", index=False, encoding="utf8")

KeyError: "['Category'] not in index"

In [15]:
import pandas as pd

# ---- 1. Chargement des données ----

# Fichier contenant les phrases segmentées
df_phrases = pd.read_csv("./artifacts/bertopic/reviews_phrases_with_topics.csv")

# Fichier clusters annotés
df_clusters = pd.read_excel("./artifacts/bertopic/topics_top_words_phrases_annoté.xlsx")

replace_map = {
    # qualité produit
    "Qualité Produit": "qualité produit",
    "Qualité produit": "qualité produit",

    # livraison
    "Service Livraison": "service livraison",
    "Service livraison": "service livraison",

    # service client
    "Service Client": "service client",
    "Service client": "service client",
}

def normalize_category(cat):
    if pd.isna(cat):
        return None
    cat = cat.strip().lower()
    return replace_map.get(cat, cat)

df_clusters["Catégorie"] = df_clusters["Catégorie"].apply(normalize_category)

# ---- 2. Préparation du fichier des phrases ----

# Fusion sur le numéro de topic
df_phrases_merged = df_phrases.merge(
    df_clusters[["Topic", "Catégorie"]],
    left_on='topics',
    right_on='Topic',
    how='left'
).drop(columns=['Topic'])

# Retirer les neutres si nécessaire
df_phrases_merged = df_phrases_merged[df_phrases_merged['Catégorie'] != 'neutre']

# Catégories possibles
categories = ["qualité produit", "service livraison", "service client"]

# Création colonnes one-hot binaires
for cat in categories:
    df_phrases_merged[cat] = (df_phrases_merged['Catégorie'] == cat).astype(int)


# ---- 3. Construction du fichier d'avis (non segmentés) ----
# On regroupe par comment_id, mais un avis peut contenir plusieurs catégories
# → On prend le max() pour chaque catégorie (si une phrase du texte l’a, l’avis l’a aussi)

# df_avis = (
#     df_phrases_merged.groupby(
#         ["comment_id", "Commentaire", "star", "date", "client",
#          "reponse", "source", "company", "ville", "maj",
#          "date_commande", "ecart", "clean_comment"]
#     )[categories]
#     .max()
#     .reset_index()
# )
# df_avis


# ---- 4. Export des fichiers ----

# # Fichier des phrases
# df_phrases_merged.to_csv(
#     "../resultats/bertopic/data/dataset_labellise_phrases.csv",
#     index=False, encoding="utf8"
# )

# # Fichier des avis complets
# df_avis.to_csv(
#     "../resultats/bertopic/data/dataset_labellise_avis.csv",
#     index=False, encoding="utf8"
# )


['service livraison' 'service client' 'qualité produit'
 'qualité produit  / service livraison' '3 thèmes'
 'service client / service livraison' 'neutre'
 'qualité produit / service livraison' 'service client/livraison'
 'service client /service livraison' 'qualité produit/ service livraison'
 'service livraison / qualité produit' 'service livraison/qualité produit'
 'service livraison/qualité produit/neutre'
 'service livraison/service client' 'service client / livraison'
 'qualité produit / livraison' None]


,comment_id,Commentaire,star,date,client,reponse,source,company,ville,maj,date_commande,ecart,clean_comment,qualité produit,service livraison,service client
0,2498,Très déçue de la qualité des articles . Après ...,2,2021-03-19,Maya L .,"Bonjour , Je suis sincèrement navré d'apprendr...",TrustedShop,ShowRoom,Montigny le bretonneux,2021-04-30,2021-02-20,27.0,très déçue de la qualité des articles . après ...,1,0,0


In [16]:
df_phrases_merged

,Commentaire,star,date,client,reponse,source,company,ville,maj,date_commande,ecart,clean_comment,comment_id,sentence,topics,Catégorie,qualité produit,service livraison,service client
4,"Bonjour , Ca doit faire 5 ans environ que je s...",1,2021-06-20 00:00:00+00:00,AUDREY Du 62,NaN,TrustPilot,ShowRoom,NaN,NaN,NaN,NaN,"bonjour , ca doit faire 5 ans environ que je s...",0,"» 89,99€… annulé 1 mois après …la 2e commande ...",104,service client/livraison,0,0,0
7,"Bonjour , Ca doit faire 5 ans environ que je s...",1,2021-06-20 00:00:00+00:00,AUDREY Du 62,NaN,TrustPilot,ShowRoom,NaN,NaN,NaN,NaN,"bonjour , ca doit faire 5 ans environ que je s...",0,au final je vais juste rester fidèle sur n aut...,197,3 thèmes,0,0,0
8,"Bonjour , Ca doit faire 5 ans environ que je s...",1,2021-06-20 00:00:00+00:00,AUDREY Du 62,NaN,TrustPilot,ShowRoom,NaN,NaN,NaN,NaN,"bonjour , ca doit faire 5 ans environ que je s...",0,! très décevant … plus du tout confiance je ne...,185,3 thèmes,0,0,0
9,"Bonjour , Ca doit faire 5 ans environ que je s...",1,2021-06-20 00:00:00+00:00,AUDREY Du 62,NaN,TrustPilot,ShowRoom,NaN,NaN,NaN,NaN,"bonjour , ca doit faire 5 ans environ que je s...",0,afin d ’ éviter que d ’ autres personnes conna...,161,3 thèmes,0,0,0
10,Vente lacoste article manquant photo prise sur...,1,2021-06-20 00:00:00+00:00,Nanasky De Verteuil,NaN,TrustPilot,ShowRoom,NaN,NaN,NaN,NaN,vente lacoste article manquant photo prise sur...,1,vente lacoste article manquant photo prise sur...,128,3 thèmes,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37621,Je ne sais pas si VP cherche à vendre ou à fai...,1,2015-10-21 00:00:00+00:00,Anne laure,NaN,TrustPilot,VeePee,NaN,NaN,NaN,NaN,je ne sais pas si vp cherche à vendre ou à fai...,15074,"je suis en litige avec eux , ils ne veulent pa...",18,3 thèmes,0,0,0
37623,Je suis client sur ce site depuis plusieurs an...,5,2015-10-02 00:00:00+00:00,Thomas GUILLAUME,NaN,TrustPilot,VeePee,NaN,NaN,NaN,NaN,je suis client sur ce site depuis plusieurs an...,15075,les marques sont variés et concernent des prod...,145,qualité produit,1,0,0
37624,Je suis client sur ce site depuis plusieurs an...,5,2015-10-02 00:00:00+00:00,Thomas GUILLAUME,NaN,TrustPilot,VeePee,NaN,NaN,NaN,NaN,je suis client sur ce site depuis plusieurs an...,15075,les prix sont parfois aléatoires ( de la tres ...,61,qualité produit,1,0,0
37625,Je suis client sur ce site depuis plusieurs an...,5,2015-10-02 00:00:00+00:00,Thomas GUILLAUME,NaN,TrustPilot,VeePee,NaN,NaN,NaN,NaN,je suis client sur ce site depuis plusieurs an...,15075,certes les délais sont parfois longs,88,qualité produit,1,0,0


### Visualisation

In [9]:
# CHARGER LES MODELES ENREGISTRES
models = {}
for name in ["français", "multilingue", "multilingue_2"]:
    with open(os.path.join("../models/topic_modeling/", f"bertopic_{name}_phrases.pkl"), 'rb') as f:
        models[name] = pickle.load(f)

In [10]:
all_topics_words = {}
topic_model = models["multilingue_2"]

for topic_id in topic_model.get_topics().keys():
    if topic_id == -1:
        continue  # ignorer les outliers
    all_topics_words[topic_id] = [word for word, _ in topic_model.get_topic(topic_id)]

all_topics_words

{1: ['avant date',
  'date',
  'date prévue',
  'avant date prévue',
  'avant',
  'prévue commande',
  'prévue',
  'date prévue commande',
  'arrivé avant date',
  'arrivé avant',
  'prévue commande reçue',
  'livraison avant',
  'arrivée',
  'reçue',
  'commande reçue'],
 2: ['livraison plus',
  'plus rapide',
  'livraison plus rapide',
  'prévu livraison plus',
  'plus rapide prévue',
  'rapide prévue',
  'plus tôt',
  'prévu livraison',
  'tôt',
  'prévu',
  'plus tôt prévu',
  'tôt prévu',
  'prévue',
  'rapide',
  'prévue livraison plus'],
 3: ['bien passé tout',
  'passé tout',
  'tout très bien',
  'bien passé',
  'tout très',
  'très bien passé',
  'très bien',
  'passé tout très',
  'déroulé',
  'bien déroulé',
  'très bien ensemble',
  'passé',
  'bien ensemble',
  'bien déroulé tout',
  'déroulé tout très'],
 4: ['recommande recommande',
  'recommande',
  'recommande recommande recommande',
  'recommande recommande tout',
  'recommande tout recommande',
  'tout recommande',


In [17]:
# topics trouvés
models["français"].get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,1,133,1_belles_très belles_beaux_qualités,"[belles, très belles, beaux, qualités, robes, ...",[les chaussures sont de très mauvaises qualité...
1,2,79,2_description_conforme_conforme description_co...,"[description, conforme, conforme description, ...","[conforme à la description ., conforme à la de..."
2,3,63,3_rien dire_dire_rien_parfait rien,"[rien dire, dire, rien, parfait rien, parfait ...","[parfait , rien à dire, parfait rien a dire ....."
3,4,51,4_voleurs_escrocs_voleurs voleurs_menteurs,"[voleurs, escrocs, voleurs voleurs, menteurs, ...","[vous êtes des voleurs ! !, ce sont des voleur..."
4,5,50,5_tout parfait_parfait_parfait tout_tout,"[tout parfait, parfait, parfait tout, tout, to...","[tout à été parfait, tout a été parfait !, tou..."
...,...,...,...,...,...
240,241,256,241_qualité_très_rapide_bonne,"[qualité, très, rapide, bonne, bonne qualité, ...",NaN
241,242,137,242_service client_client_service_joindre,"[service client, client, service, joindre, sav...",NaN
242,243,164,243_attends_commandé_attends remboursement_aime,"[attends, commandé, attends remboursement, aim...",NaN
243,244,78,244_correspondent_conformes_commandés_produits,"[correspondent, conformes, commandés, produits...",NaN


In [18]:
# topics trouvés
models["multilingue"].get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,1,205,1_couleur_couleurs_bonne couleur_coloris,"[couleur, couleurs, bonne couleur, coloris, ph...",[j ai commander une parure et la couleur reçu ...
1,2,174,2_connectée_fonctionne_commandé_acheté,"[connectée, fonctionne, commandé, acheté, reçu...","[site a évité ..... depuis veepee , les problè..."
2,3,136,3_balmain_soleil_guess_lunette,"[balmain, soleil, guess, lunette, jolies, elle...","[bonjour , j ’ ai fait un achat de deux paires..."
3,4,117,4_étoile_mettre_mets_mets étoile,"[étoile, mettre, mets, mets étoile, étoile car...",[je mets une étoile car il est impossible d'en...
4,5,83,5_français_france_française_peine français,"[français, france, française, peine français, ...","[je contacte le service après vente de vp , qu..."
...,...,...,...,...,...
209,210,401,210_remboursement_toujours_retour_compte,"[remboursement, toujours, retour, compte, aprè...",NaN
210,211,149,211_parfait_tout_parfaitement_tout parfait,"[parfait, tout, parfaitement, tout parfait, au...",NaN
211,212,148,212_marchandise_produits_produit_conforme,"[marchandise, produits, produit, conforme, pro...",NaN
212,213,195,213_site_plus site_plus_commander site,"[site, plus site, plus, commander site, comman...",NaN


In [19]:
# topics trouvés
models["multilingue_2"].get_topic_info().to_csv("./artifacts/bertopic/topics_top_words_phrases.csv", index=False, encoding="utf8")
models["multilingue_2"].get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,1,117,1_ete_tres_meme_apres,"[ete, tres, meme, apres, probleme, recu, malgr...",[commande de bottes en taille 35 et reçue du 3...
1,2,116,2_satisfaite_très satisfaite_satisfaite comman...,"[satisfaite, très satisfaite, satisfaite comma...","[je suis très satisfaite !, je suis très satis..."
2,3,108,3_livraison peu_peu long_livraison peu long_long,"[livraison peu, peu long, livraison peu long, ...","[délai livraison un peu long ., délai de livra..."
3,4,107,4_conforme_description_attentes_conformes,"[conforme, description, attentes, conformes, c...","[conforme à la description, conforme à la desc..."
4,5,90,5_cliente_années_cliente depuis_depuis,"[cliente, années, cliente depuis, depuis, anné...","[je suis cliente depuis des années ., je suis ..."
...,...,...,...,...,...
393,394,128,394_rapide_livraison rapide_top_rapide livraison,"[rapide, livraison rapide, top, rapide livrais...",NaN
394,395,207,395_satisfaite_qualité_rapide_bonne qualité,"[satisfaite, qualité, rapide, bonne qualité, t...",NaN
395,396,95,396_cliente_terminé_absolument plus_impression...,"[cliente, terminé, absolument plus, impression...",NaN
396,397,174,397_carton_scotch_abîmé_plastique,"[carton, scotch, abîmé, plastique, protection,...",NaN


In [57]:
# Mots-clés associés à un topic
models["français"].get_topic(1)

[('fuir', np.float64(0.009360780014971159)),
 ('voleurs', np.float64(0.006536055892552369)),
 ('mefiez site', np.float64(0.006482369029416102)),
 ('deux pieds', np.float64(0.006482369029416102)),
 ('déconseille', np.float64(0.006154332195534195)),
 ('euros très', np.float64(0.006110943383651499)),
 ('mefiez', np.float64(0.006110943383651499)),
 ('honte', np.float64(0.005802463867721284)),
 ('moque monde', np.float64(0.005643290670603633)),
 ('site fuir', np.float64(0.005350966900032679)),
 ('déconseille fortement', np.float64(0.005323751485776222)),
 ('bravo', np.float64(0.005034728530178279)),
 ('chemin', np.float64(0.005013290442626738)),
 ('site', np.float64(0.004958475523537399)),
 ('garantie', np.float64(0.0049334318566711605))]

In [58]:
# Mots-clés associés à un topic
models["multilingue"].get_topic(1)

[('euros', np.float64(0.014610914133020642)),
 ('50', np.float64(0.007289140282516761)),
 ('bon achat', np.float64(0.005672219830212208)),
 ('frais', np.float64(0.005338886535221998)),
 ('payé', np.float64(0.005196259268486484)),
 ('article', np.float64(0.005114712229288268)),
 ('99', np.float64(0.005028546320017942)),
 ('retour', np.float64(0.004866339700676376)),
 ('achat', np.float64(0.004624742993202379)),
 ('euro', np.float64(0.004410232866890986)),
 ('frais retour', np.float64(0.004264238898192449)),
 ('remboursement', np.float64(0.0037714733604076764)),
 ('payer', np.float64(0.003715378032467817)),
 ('bon', np.float64(0.0036115305747170686)),
 ('90', np.float64(0.0036032736458079494))]

In [11]:
fig1 = models["français"].visualize_topics(top_n_topics=10)
fig2 = models["multilingue_2"].visualize_topics(top_n_topics=10)

combined_fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Modèle 1 : Français", "Modèle 2 : Multilingue")
)

for trace in fig1['data']:
    combined_fig.add_trace(trace, row=1, col=1)

for trace in fig2['data']:
    combined_fig.add_trace(trace, row=1, col=2)

combined_fig.update_layout(
    title_text="Répartition des topics",
    showlegend=False,
    height=600,
    width=1000
)

combined_fig.show()

IndexError: index 313 is out of bounds for axis 0 with size 306

In [12]:
models["français"].visualize_barchart(top_n_topics=10)

In [13]:
models["multilingue_2"].visualize_barchart(top_n_topics=10)

In [14]:
models["français"].visualize_hierarchy()

In [15]:
models["multilingue_2"].visualize_hierarchy()